In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/AI systems Project/SummarIQ')

In [3]:
cd /content/drive/MyDrive/AI systems Project/SummarIQ

/content/drive/MyDrive/AI systems Project/SummarIQ


In [17]:
pip install transformers datasets sentencepiece torch regex

In [18]:
from datasets import load_dataset
import re
from transformers import pipeline
import nltk
nltk.download("punkt")
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from transformers import BartTokenizer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [21]:
raw_dataset = load_dataset("ccdv/arxiv-summarization")
train_data = raw_dataset["train"].train_test_split(test_size=0.2, shuffle=True)["train"]
val_data = raw_dataset["validation"].train_test_split(test_size=0.2, shuffle=True)["train"]

print("Train size:", len(train_data))
print("Validation size:", len(val_data))

Train size: 162429
Validation size: 5148


In [22]:
def extract_section(text, section_name):
    pattern = rf"\\section\{{.*?{section_name}.*?\}}(.*?)(?=\\section|\Z)"
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else ""

In [23]:
def extract_equations(text, max_eq=6):
    return re.findall(
        r'\$.*?\$|\\begin\{equation\}.*?\\end\{equation\}',
        text,
        re.DOTALL
    )[:max_eq]

In [24]:
method_keywords = [
    "methodology", "methods", "approach", "proposed method",
    "model", "technique", "experimental setup"
]

def extract_methodology(text, max_sentences=6):
    sentences = sent_tokenize(text)

    method_sents = []

    science_verbs = [
        r"we propose", r"we develop", r"we introduce",
        r"our model", r"our approach", r"this method",
        r"this model", r"our technique", r"we design"
    ]

    # Extract sentences containing section-like keywords
    for s in sentences:
        if any(re.search(rf"(?i){kw}", s) for kw in method_keywords):
            method_sents.append(s)

    # Extract methodological sentences if section not found
    if len(method_sents) < 3:
        for s in sentences:
            if any(re.search(v, s, re.IGNORECASE) for v in science_verbs):
                method_sents.append(s)

    blacklist = ["related work", "introduction", "references", "\\section"]
    method_sents = [
        s for s in method_sents
        if not any(bl in s.lower() for bl in blacklist)
    ]

    if not method_sents:
        return "No clear methodology — inferred approach from context"

    return " ".join(method_sents[:max_sentences])

In [25]:
def extract_results(text):
    patterns = [r"(?i)result", r"(?i)experiment", r"(?i)evaluation"]
    sentences = sent_tokenize(text)

    result_lines = []
    for s in sentences:
        if any(re.search(p, s) for p in patterns):
            result_lines.append(s)

    if not result_lines:
        return "No clear results"

    return " ".join(result_lines[:5])

In [26]:
def build_structured(batch):
    article = batch["article"]

    return {
        "text": article,
        "core": batch["abstract"],
        "method": extract_methodology(article),
        "equations": " | ".join(extract_equations(article)),
        "results": extract_results(article)
    }

In [27]:
## Don't need to execute this cell the datasets already saved just used it

processed_train = train_data.map(build_structured)
processed_val = val_data.map(build_structured)

Map:   0%|          | 0/162429 [00:00<?, ? examples/s]

Map:   0%|          | 0/5148 [00:00<?, ? examples/s]

In [31]:
processed_train.save_to_disk("/content/drive/MyDrive/AI systems Project/SummarIQ/processed_train")
processed_val.save_to_disk("/content/drive/MyDrive/AI systems Project/SummarIQ/processed_val")

Saving the dataset (0/25 shards):   0%|          | 0/162429 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5148 [00:00<?, ? examples/s]

In [28]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [33]:
def preprocess(batch):
    source = (
        "Summarize into structured sections:\n\n"
        "Core Idea:\nMethodology:\nKey Equation(s):\nResults:\n\n"
        "Text:\n" + batch["text"]
    )

    target = (
        f"Core Idea: {batch['core']}\n"
        f"Methodology: {batch['method']}\n"
        f"Key Equation(s): {batch['equations']}\n"
        f"Results: {batch['results']}"
    )

    model_inputs = tokenizer(
        source, max_length=1024, truncation=True
    )
    labels = tokenizer(
        target, max_length=300, truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [34]:
train_dataset = processed_train.map(preprocess, batched=False)
val_dataset = processed_val.map(preprocess, batched=False)

Map:   0%|          | 0/162429 [00:00<?, ? examples/s]

Map:   0%|          | 0/5148 [00:00<?, ? examples/s]